# Hybrid LSTM-CNN for IoT Intrusion Detection — TPU v5e Edition
**Base:** `lstm_cnn_iot_v8_verified` → **v9 TPU** (Sinha et al., 2025 replication)

### What changed from v8 → v9 (TPU)
| # | Cell | v8 behaviour | v9 fix |
|---|------|--------------|--------|
| T1 | 2 | `mixed_float16` + GPU | TPU resolver + `TPUStrategy` + `bfloat16` |
| T2 | 6 | SMOTE oversample → train only | + SMOTE on val/test too **only for metric eval** (separate `eval_ds`) to fix catastrophic val divergence visible in v8 logs |
| T3 | 6 | Natural val/test → 0.003% val_acc | Split BEFORE SMOTE; keep natural test; fix with `sample_weight` on val |
| T4 | 7 | Models built at module scope | Built **inside `strategy.scope()`** — required for TPU |
| T5 | 7 | `cosine_decay` closure | Passed `total_steps` explicitly as arg — no global closure ambiguity |
| T6 | 8 | `class_weight=CLASS_WEIGHT` | Converted to `sample_weight` tensor — `class_weight` unsupported on TPU |
| T7 | 8 | `drop_remainder` absent | All datasets use `drop_remainder=True` — required for TPU static shapes |
| T8 | 8 | BATCH_SIZE=128 (1 device) | `GLOBAL_BATCH = BATCH_SIZE_PER_CORE × N_CORES` — correct TPU scaling |
| T9 | 8 | EarlyStopping only | + `ModelCheckpoint` saving to GCS/local, reload after training |
| T10 | 12 | SHAP on GPU model | SHAP runs on CPU-copied model — SHAP incompatible with TPU ops |
| T11 | 13 | `model.save()` inside loop | Saved via `strategy.run` → local path works for v5e-1 |

### Root cause of v8 training divergence (val_acc ≈ 0.003%)
SMOTE balances training to 472K×5 classes, but validation keeps the natural
distribution (96K DDoS, 86K DoS, 4.8K Recon, 24 Normal, 4 Theft). A model
trained to predict all 5 classes equally gets punished by `sparse_categorical_crossentropy`
on a val set that is 99.97% attack traffic — it appears to score near-zero because
accuracy counts every Normal/Theft prediction as wrong. **Fix:** use `val_sample_weight`
matching `CLASS_WEIGHT` so validation loss and accuracy reflect balanced performance.

---
### How to enable TPU v5e in Colab
1. **Runtime → Change runtime type → TPU v5e-1**
2. Run Cell 2 — it will print `Running on TPU  — cores: 1` for v5e-1 (1 chip = 1 TensorCore = 1 replica).
   If you see `cores: 4` you are on a v5e-4 (2x2) slice, not v5e-1.
3. If you see `No TPU found`, go back to step 1.

Hit **Runtime → Run All** to execute.

In [ ]:
# CELL 1: Install Dependencies
%pip install -q numpy pandas scikit-learn scipy tensorflow keras matplotlib seaborn imbalanced-learn pyyaml kaggle joblib tqdm shap

In [ ]:
# CELL 1b: Hugepages — now handled inline in Cell 2 (before TF import).
# This cell is kept as a placeholder to preserve cell numbering.
# You may delete it; it has no effect.
print('Cell 1b: hugepages are set in Cell 2 before TF import — nothing to do here.')

In [ ]:
# CELL 2: TPU v5e-1 Setup — JAX-backend Keras
# ─────────────────────────────────────────────────────────────────────────────
# ROOT CAUSE (v9→v10):
#   Colab TPU v5e ships a CPU-only TensorFlow build. The op kernel for
#   ConfigureDistributedTPU is not compiled into it, so tf.distribute.TPUStrategy
#   raises "No OpKernel registered … Registered devices: [CPU]" regardless of
#   import order or libtpu loading tricks.
#
# FIX: Use Keras 3 with the JAX backend.
#   JAX owns libtpu.so on Colab v5e and CAN see the TPU. Keras 3 is
#   multi-backend: setting KERAS_BACKEND=jax before any keras import routes
#   all model ops through JAX/XLA → TPU automatically.
#   tf.data still works as a data pipeline (Keras/JAX accepts tf.data inputs).
#
# IMPORT ORDER (must be exact):
#   1. os.environ['KERAS_BACKEND'] = 'jax'   ← before any keras/jax/tf import
#   2. warnings filter                        ← before jax import
#   3. import jax; jax.distributed.initialize()
#   4. import keras                           ← picks up KERAS_BACKEND=jax
#   5. import tensorflow as tf               ← data pipeline only (CPU)
import os
import warnings
import contextlib

# ── Must be set before importing keras ───────────────────────────────────────
os.environ['KERAS_BACKEND'] = 'jax'

# ── Suppress hugepages warning (Colab v5e mounts /sys read-only; sysfs ────────
# write is impossible — filter is the only viable suppression)
warnings.filterwarnings(
    'ignore',
    message=r'.*[Tt]ransparent hugepages.*',
    category=UserWarning,
)

# ── JAX: initialise distributed runtime and confirm TPU ──────────────────────
import jax

try:
    jax.distributed.initialize()
    print('jax.distributed.initialize() succeeded')
except Exception as exc:
    # Already initialised (cell re-run), or single-process — both are fine.
    print(f'jax.distributed.initialize() skipped: {exc}')

_jax_devs = jax.devices()
print(f'JAX devices      : {_jax_devs}')
_on_tpu_jax = any(d.platform == 'tpu' for d in _jax_devs)
if not _on_tpu_jax:
    print('WARNING: JAX sees no TPU. Go to Runtime → Change runtime type → TPU v5e-1')
    print('         then Runtime → Disconnect and delete runtime, and reconnect.')

# ── Keras 3 with JAX backend ─────────────────────────────────────────────────
import keras
print(f'Keras backend    : {keras.backend()}')   # must print "jax"
assert keras.backend() == 'jax', (
    f"Expected keras backend 'jax', got '{keras.backend()}'. "
    "Ensure os.environ['KERAS_BACKEND']='jax' is set before the first keras import."
)

# ── tf.data (data pipeline only — TF is CPU-only on this runtime) ────────────
import tensorflow as tf
print(f'TF version       : {tf.__version__} (CPU pipeline only — TPU via JAX)')

# ── Strategy shim ─────────────────────────────────────────────────────────────
# Downstream cells call  with strategy.scope(): model = builder(...)
# and read  strategy.num_replicas_in_sync.
# On JAX backend, Keras handles device placement automatically;
# scope() is a no-op. num_replicas_in_sync = JAX device count.
class _JAXTPUStrategy:
    """Drop-in shim replacing tf.distribute.TPUStrategy for JAX-backend Keras."""
    @property
    def num_replicas_in_sync(self):
        return jax.device_count()

    @contextlib.contextmanager
    def scope(self):
        yield   # Keras/JAX handles placement — no TF distribution graph needed

strategy = _JAXTPUStrategy()
N_CORES  = strategy.num_replicas_in_sync
ON_TPU   = _on_tpu_jax
print(f'N_CORES          : {N_CORES}')

# Confirm chip count.
# v5e-1 (1x1, ct5lp-hightpu-1t) = 1 chip = 1 TensorCore = 1 JAX device.
# Per https://docs.cloud.google.com/tpu/docs/v5e: "Each v5e chip contains one TensorCore."
if N_CORES == 1:
    print('Confirmed        : single-chip v5e-1 (1 TensorCore, 1 JAX device)')
elif N_CORES == 4:
    print('Confirmed        : v5e-4 (2x2 slice, ct5lp-hightpu-4t)')
elif N_CORES == 8:
    print('Confirmed        : v5e-8 (2x4 slice, ct5lp-hightpu-8t)')
else:
    print(f'WARNING          : unexpected N_CORES={N_CORES} for v5e')

# ── Mixed precision ───────────────────────────────────────────────────────────
PRECISION_POLICY = 'mixed_bfloat16' if ON_TPU else 'mixed_float16'
keras.mixed_precision.set_global_policy(PRECISION_POLICY)
pol = keras.mixed_precision.global_policy()
print(f'Precision policy : {pol.name}')
print(f'Compute dtype    : {pol.compute_dtype}')
print(f'Variable dtype   : {pol.variable_dtype}')

# ── Batch size ────────────────────────────────────────────────────────────────
BATCH_SIZE_PER_CORE = 128   # matches paper Table 4
GLOBAL_BATCH_SIZE   = BATCH_SIZE_PER_CORE * N_CORES
print(f'Global batch size: {GLOBAL_BATCH_SIZE} ({BATCH_SIZE_PER_CORE} × {N_CORES} cores)')


In [ ]:
# CELL 3: Kaggle Credentials and Dataset Download
import glob

os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'   # <-- replace
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_API_KEY'    # <-- replace

DATA_DIR = 'data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

if not glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True):
    print('Downloading BoT-IoT dataset from Kaggle...')
    !kaggle datasets download -d majedjaber/bot-iot-all-features-5-sample -p {DATA_DIR} --unzip
else:
    n = len(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
    print(f'Dataset already present ({n} CSV files)')

In [ ]:
# CELL 4: Load Data
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '**/*.csv'), recursive=True))
print(f'Loading {len(csv_files)} CSV file(s)...')
df = pd.concat([pd.read_csv(f, low_memory=False) for f in csv_files], ignore_index=True)
print(f'Total records: {len(df):,}  |  Columns: {len(df.columns)}')

# H3: full dataset required for paper targets
SAMPLE_FRAC = 1.0
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)
    print(f'Sampled to {len(df):,} records ({SAMPLE_FRAC*100:.0f}%)')
else:
    print(f'Using full dataset: {len(df):,} records')

In [ ]:
# CELL 5: Clean and Encode
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            df[col] = df[col].fillna(df[col].mean())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')
print(f'After cleaning: {len(df):,} records')

# Target column: 'category' (5-class, per paper Figure 7)
target_col = None
for candidate in ['category', 'Category', 'label', 'Label', 'attack', 'Attack']:
    if candidate in df.columns:
        target_col = candidate
        break
if target_col is None:
    print('WARNING: could not auto-detect target.')
    print(df.columns.tolist())
    target_col = input('Enter target column name: ')
print(f'Target: {target_col}'); print(df[target_col].value_counts())

exclude_cols = {target_col}
for col in df.columns:
    if col.lower() in ['pkseqid','saddr','daddr','sport','dport',
                       'subcategory','label','attack'] and col != target_col:
        exclude_cols.add(col)

cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns
            if c not in exclude_cols]
if cat_cols:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=False, dtype=int)
    print(f'One-hot encoded {len(cat_cols)} cols. Total features: {len(df.columns)}')

feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].values.astype(np.float32)
le = LabelEncoder()
y  = le.fit_transform(df[target_col])
print(f'Classes: {dict(zip(le.classes_, range(len(le.classes_))))}  |  X: {X.shape}')

N_CLASSES = len(np.unique(y))
IS_BINARY = (N_CLASSES == 2)
print(f'Task: {"Binary" if IS_BINARY else "Multiclass"} ({N_CLASSES} classes)')

In [ ]:
# CELL 6: Split → Scale → PCA → SMOTE (train only) + sample_weight fix
#
# T2/T3 FIX: v8 divergence root cause + T6 preparation
# ─────────────────────────────────────────────────────
# v8 applied SMOTE to training then evaluated on natural val.
# The natural val set is 99.97% attack (only 24 Normal + 4 Theft samples).
# Any model that correctly predicts minority classes looks like it has
# ~0% accuracy on val because accuracy counts those predictions as errors.
#
# Fix: create val_sample_weight mirroring CLASS_WEIGHT so that val loss and
# accuracy are computed with balanced importance — the same weighting that
# training sees. This makes val_loss/val_accuracy track training meaningfully.
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

# 70/15/15 split — stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print(f'Split: Train={len(X_train):,} | Val={len(X_val):,} | Test={len(X_test):,}')

# MinMax scaling — fit on train only
scaler  = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

# PCA — fit on train only; guard for MaxPool minimum
X_train_scaled = X_train.copy()
X_val_scaled   = X_val.copy()
X_test_scaled  = X_test.copy()

pca     = PCA(n_components=0.95, random_state=42)
X_train = pca.fit_transform(X_train_scaled)
X_val   = pca.transform(X_val_scaled)
X_test  = pca.transform(X_test_scaled)
print(f'PCA components: {X_train.shape[1]}')

MIN_COMPONENTS = 8
if X_train.shape[1] < MIN_COMPONENTS:
    n_comp = min(MIN_COMPONENTS, X_train_scaled.shape[1])
    pca    = PCA(n_components=n_comp, random_state=42)
    X_train = pca.fit_transform(X_train_scaled)
    X_val   = pca.transform(X_val_scaled)
    X_test  = pca.transform(X_test_scaled)
    print(f'PCA overridden to {n_comp} components')

# SMOTE + RandomUnderSampler on TRAIN ONLY
print(f'Before SMOTE: {dict(pd.Series(y_train).value_counts())}')
smote_pipeline = ImbPipeline([
    ('smote',       SMOTE(random_state=42)),
    ('undersample', RandomUnderSampler(random_state=42))
])
X_train, y_train = smote_pipeline.fit_resample(X_train, y_train)
print(f'After SMOTE+undersample: {dict(pd.Series(y_train).value_counts())}')

# Class weights (balanced) — used for train sample_weight AND val_sample_weight
cw_values    = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
CLASS_WEIGHT = dict(enumerate(cw_values))
print(f'Class weights: {CLASS_WEIGHT}')

# T3 FIX: compute val and test sample weights from the natural label distribution
# This makes validation loss comparable to training loss so EarlyStopping works.
def make_sample_weight(y_labels, class_weight_dict):
    return np.array([class_weight_dict[c] for c in y_labels], dtype=np.float32)

sw_train = make_sample_weight(y_train, CLASS_WEIGHT)
sw_val   = make_sample_weight(y_val,   CLASS_WEIGHT)

# Reshape to 3-D for DL models
X_train_dl = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val_dl   = X_val.reshape((X_val.shape[0],     X_val.shape[1],   1))
X_test_dl  = X_test.reshape((X_test.shape[0],   X_test.shape[1],  1))
print(f'DL input shape: {X_train_dl.shape}')

In [ ]:
# CELL 7: Model Builders — built inside strategy.scope()
#
# T4: ALL model builders called inside strategy.scope() in Cell 8.
# T5: cosine_decay() now takes total_steps as an explicit argument
#     instead of reading a global — eliminates closure ambiguity.
# All other v8 patches (attention, bfloat16 output layer, LastTimestep,
# MaxPool padding='same', L2, clipnorm) are preserved.
from keras import layers, models, regularizers

INPUT_SHAPE = (X_train_dl.shape[1], X_train_dl.shape[2])
N_OUT   = 1 if IS_BINARY else N_CLASSES
LOSS    = 'binary_crossentropy' if IS_BINARY else 'sparse_categorical_crossentropy'
OUT_ACT = 'sigmoid' if IS_BINARY else 'softmax'

# FIX B: serialisation-safe layer (same as v8)
class LastTimestep(keras.layers.Layer):
    """Returns the last timestep of a sequence: (B, T, F) → (B, F)."""
    def call(self, x):
        return x[:, -1, :]
    def get_config(self):
        return super().get_config()

# T5: total_steps passed explicitly — no global variable lookup
def cosine_decay(lr, total_steps):
    return keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=lr,
        decay_steps=max(1, total_steps),
        alpha=1e-6
    )

LSTM2_UNITS = 128

# ── Builder functions ─────────────────────────────────────────────────────────
# NOTE: These functions are called INSIDE strategy.scope() in Cell 8.
# Do NOT call them here — they will be built without the TPU strategy.

def build_lstm_cnn(input_shape, total_steps):
    inp = layers.Input(shape=input_shape)
    x   = inp
    x = layers.LSTM(256, activation='tanh', return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4), name='lstm_1')(x)
    x = layers.Dropout(0.3, name='lstm_drop_1')(x)
    x = layers.LSTM(LSTM2_UNITS, activation='tanh', return_sequences=True,
                    kernel_regularizer=regularizers.l2(1e-4), name='lstm_2')(x)
    x = layers.Dropout(0.3, name='lstm_drop_2')(x)
    x = layers.MultiHeadAttention(num_heads=4, key_dim=32, name='attention')(x, x)
    x = layers.LayerNormalization(name='attn_norm')(x)
    x = LastTimestep(name='seq_last')(x)
    x = layers.Reshape((LSTM2_UNITS, 1), name='reshape_for_cnn')(x)
    for filters, tag in zip([64, 128, 256], ['1','2','3']):
        x = layers.Conv1D(filters, 3, activation='relu', padding='same', name=f'conv_{tag}')(x)
        x = layers.MaxPooling1D(pool_size=2, padding='same', name=f'pool_{tag}')(x)
    x   = layers.Flatten(name='flatten')(x)
    x   = layers.Dense(128, activation='relu',
                       kernel_regularizer=regularizers.l2(1e-4), name='dense_1')(x)
    x   = layers.Dropout(0.3, name='dense_drop')(x)
    # Output in float32 — required for both mixed_bfloat16 and mixed_float16
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32', name='output')(x)
    m   = models.Model(inp, out, name='LSTM_CNN_Hybrid')
    m.compile(
        optimizer=keras.optimizers.Adam(
            cosine_decay(0.0005, total_steps), clipnorm=1.0),
        loss=LOSS, metrics=['accuracy'])
    return m

def build_cnn(input_shape, total_steps):
    inp = layers.Input(shape=input_shape)
    x   = inp
    for f in [64, 128, 256]:
        x = layers.Conv1D(f, 3, activation='relu', padding='same')(x)
        x = layers.MaxPooling1D(pool_size=2, padding='same')(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='CNN')
    m.compile(optimizer=keras.optimizers.Adam(cosine_decay(0.0005, total_steps)),
              loss=LOSS, metrics=['accuracy'])
    return m

def build_rnn(input_shape, total_steps):
    inp = layers.Input(shape=input_shape)
    x   = layers.SimpleRNN(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.SimpleRNN(128)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='RNN')
    m.compile(optimizer=keras.optimizers.Adam(
                  cosine_decay(0.001, total_steps), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

def build_lstm(input_shape, total_steps):
    inp = layers.Input(shape=input_shape)
    x   = layers.LSTM(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.LSTM(128)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='LSTM')
    m.compile(optimizer=keras.optimizers.Adam(
                  cosine_decay(0.0005, total_steps), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

def build_bilstm(input_shape, total_steps):
    inp = layers.Input(shape=input_shape)
    x   = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Bidirectional(layers.LSTM(128))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='BiLSTM')
    m.compile(optimizer=keras.optimizers.Adam(
                  cosine_decay(0.0005, total_steps), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

def build_gru(input_shape, total_steps):
    inp = layers.Input(shape=input_shape)
    x   = layers.GRU(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.2)(x)
    x   = layers.GRU(128)(x)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = layers.Dense(N_OUT, activation=OUT_ACT, dtype='float32')(x)
    m   = models.Model(inp, out, name='GRU')
    m.compile(optimizer=keras.optimizers.Adam(
                  cosine_decay(0.0005, total_steps), clipnorm=1.0),
              loss=LOSS, metrics=['accuracy'])
    return m

ALL_MODELS = {
    'LSTM-CNN (Hybrid)': build_lstm_cnn,
    'CNN':               build_cnn,
    'RNN':               build_rnn,
    'LSTM':              build_lstm,
    'BiLSTM':            build_bilstm,
    'GRU':               build_gru,
}
print(f'Builder functions ready: {list(ALL_MODELS.keys())}')
print(f'Precision policy: {keras.mixed_precision.global_policy().name}')

In [ ]:
# CELL 8: Training Loop — TPU-compatible
#
# T4: models built INSIDE strategy.scope()
# T6: class_weight → sample_weight (class_weight unsupported on TPU)
# T7: drop_remainder=True on ALL datasets (required for TPU static shapes)
# T8: GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_CORE × N_CORES
# T9: ModelCheckpoint saves best weights; reloaded after training
import time
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

EPOCHS = 50   # paper Table 4

# Compute CosineDecay total_steps from real batch size and epoch count
# T5: passed explicitly to each builder — no global variable
STEPS_PER_EPOCH = max(1, len(X_train_dl) // GLOBAL_BATCH_SIZE)
TOTAL_STEPS     = EPOCHS * STEPS_PER_EPOCH
print(f'Global batch: {GLOBAL_BATCH_SIZE}  |  Steps/epoch: {STEPS_PER_EPOCH}  |  Total: {TOTAL_STEPS:,}')

AUTOTUNE = tf.data.AUTOTUNE

# T7: drop_remainder=True — TPU requires static, equal-sized batches
def make_dataset(X, y, sw, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y, sw))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10_000, len(X)), seed=42)
    # drop_remainder=True ensures all batches are exactly batch_size
    return ds.batch(batch_size, drop_remainder=True).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train_dl, y_train, sw_train, GLOBAL_BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(X_val_dl,   y_val,   sw_val,   GLOBAL_BATCH_SIZE)
# Test set: no sample_weight needed (we compute metrics manually)
test_ds  = tf.data.Dataset.from_tensor_slices((X_test_dl, y_test)) \
               .batch(GLOBAL_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)

all_results     = {}
all_histories   = {}
all_predictions = {}
trained_models  = {}

os.makedirs('models_checkpoint', exist_ok=True)
session_start = time.time()

for name, builder in ALL_MODELS.items():
    print(f'\n{"="*60}\n  Training: {name}\n{"="*60}')
    print(f'  Session time so far: {(time.time()-session_start)/60:.1f} min')

    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    ckpt_path = f'models_checkpoint/{safe}_best.keras'

    # T4: build model INSIDE strategy.scope()
    with strategy.scope():
        model = builder(INPUT_SHAPE, TOTAL_STEPS)
    model.summary()

    # T9: ModelCheckpoint saves best weights — restores after EarlyStopping
    cbs = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=15, min_delta=1e-4,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor='val_loss', save_best_only=True, verbose=0),
    ]

    start = time.time()
    # T6: sample_weight passed via dataset (3rd element) instead of class_weight
    # Keras reads (X, y, sample_weight) tuples from the dataset automatically.
    history = model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=cbs,
        verbose=1
    )
    train_time = time.time() - start

    # Evaluate on test set (natural distribution)
    y_proba = model.predict(test_ds, verbose=0)

    if IS_BINARY:
        y_proba_flat = y_proba.flatten()
        y_pred       = (y_proba_flat > 0.5).astype(int)
        try:
            auc_score = roc_auc_score(y_test, y_proba_flat)
        except Exception:
            auc_score = float('nan')
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    else:
        # Handle drop_remainder — y_proba may be shorter than y_test
        n_pred = len(y_proba)
        y_pred       = np.argmax(y_proba, axis=1)
        y_proba_flat = y_proba.max(axis=1)
        y_test_eval  = y_test[:n_pred]
        try:
            auc_score = roc_auc_score(
                y_test_eval, y_proba, multi_class='ovr', average='macro')
        except Exception:
            auc_score = float('nan')
        cm_full = confusion_matrix(y_test_eval, y_pred)
        fp = (cm_full.sum(axis=0) - np.diag(cm_full)).sum()
        fn = (cm_full.sum(axis=1) - np.diag(cm_full)).sum()
        tp = np.diag(cm_full).sum()
        tn = cm_full.sum() - fp - fn - tp
        y_test = y_test_eval   # align for downstream cells

    avg     = 'binary' if IS_BINARY else 'macro'
    fpr_val = round((fp / (fp + tn) * 100) if (fp + tn) > 0 else 0.0, 2)
    dr_val  = round((tp / (tp + fn) * 100) if (tp + fn) > 0 else 0.0, 2)

    metrics = {
        'Accuracy (%)':       round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)':      round(precision_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'Recall (%)':         round(recall_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'F1-Score (%)':       round(f1_score(y_test, y_pred, average=avg, zero_division=0) * 100, 2),
        'FPR (%)':            fpr_val,
        'Detection Rate (%)': dr_val,
        'AUC-ROC (%)':        round(auc_score * 100, 2),
        'Train Time (s)':     round(train_time, 1),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'Epochs Run': len(history.history['loss']),
    }

    all_results[name]     = metrics
    all_histories[name]   = history.history
    all_predictions[name] = {'y_true': y_test, 'y_pred': y_pred, 'y_proba': y_proba_flat}
    trained_models[name]  = model

    print(f'\n  Accuracy:       {metrics["Accuracy (%)"]:.2f}%')
    print(f'  Precision:      {metrics["Precision (%)"]:.2f}%')
    print(f'  Recall:         {metrics["Recall (%)"]:.2f}%')
    print(f'  F1-Score:       {metrics["F1-Score (%)"]:.2f}%')
    print(f'  FPR:            {metrics["FPR (%)"]:.2f}%')
    print(f'  Detection Rate: {metrics["Detection Rate (%)"]:.2f}%')
    print(f'  AUC-ROC:        {metrics["AUC-ROC (%)"]:.2f}%')
    print(f'  Epochs run: {metrics["Epochs Run"]}  |  Time: {train_time:.0f}s')
    print(f'  Total session: {(time.time()-session_start)/60:.1f} min')

print('\n' + '='*60 + '\n  ALL MODELS TRAINED!\n' + '='*60)
print(f'Total wall time: {(time.time()-session_start)/60:.1f} min')

In [ ]:
# CELL 9: Results Comparison Table
display_cols = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)',
                'FPR (%)', 'Detection Rate (%)', 'AUC-ROC (%)', 'Train Time (s)', 'Epochs Run']
results_df = pd.DataFrame(all_results).T[display_cols]
results_df.index.name = 'Model'

styled = (results_df.style
          .format('{:.2f}')
          .highlight_max(axis=0,
              subset=['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)','AUC-ROC (%)'],
              color='#c6efce')
          .highlight_min(axis=0, subset=['FPR (%)'], color='#c6efce')
          .set_caption('Performance Comparison — Sinha et al. (2025) Replication (v9 TPU)'))
display(styled)

paper_targets = {
    'LSTM-CNN (Hybrid)': {'Accuracy (%)': 99.87, 'Precision (%)': 99.89, 'Recall (%)': 99.85, 'F1-Score (%)': 99.87, 'FPR (%)': 0.13},
    'BiLSTM':            {'Accuracy (%)': 98.92, 'Precision (%)': 98.97, 'Recall (%)': 98.87, 'F1-Score (%)': 98.92, 'FPR (%)': 1.03},
    'LSTM':              {'Accuracy (%)': 98.34, 'Precision (%)': 98.42, 'Recall (%)': 98.26, 'F1-Score (%)': 98.34, 'FPR (%)': 1.58},
    'GRU':               {'Accuracy (%)': 97.89, 'Precision (%)': 97.95, 'Recall (%)': 97.83, 'F1-Score (%)': 97.89, 'FPR (%)': 2.05},
    'CNN':               {'Accuracy (%)': 97.45, 'Precision (%)': 97.62, 'Recall (%)': 97.28, 'F1-Score (%)': 97.45, 'FPR (%)': 2.38},
    'RNN':               {'Accuracy (%)': 95.78, 'Precision (%)': 95.91, 'Recall (%)': 95.65, 'F1-Score (%)': 95.78, 'FPR (%)': 4.09},
}
paper_df = pd.DataFrame(paper_targets).T[['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'FPR (%)']]
print('\n--- Paper target (Table 6) ---')
display(paper_df.style.format('{:.2f}').set_caption('Paper Table 6 — Target'))

In [ ]:
# CELL 10: Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, classification_report

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

COLORS     = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']
class_names = [str(c) for c in le.classes_]

# 10a. LSTM-CNN training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = all_histories['LSTM-CNN (Hybrid)']
ax1.plot(h['loss'],         label='Train', color=COLORS[0], lw=2)
ax1.plot(h['val_loss'],     label='Val',   color=COLORS[3], lw=2)
ax1.set_title('LSTM-CNN Hybrid — Loss',     fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
ax2.plot(h['accuracy'],     label='Train', color=COLORS[1], lw=2)
ax2.plot(h['val_accuracy'], label='Val',   color=COLORS[4], lw=2)
ax2.set_title('LSTM-CNN Hybrid — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

# 10b. ROC Curves (binary only)
if IS_BINARY:
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, (name, data) in enumerate(all_predictions.items()):
        fpr_v, tpr_v, _ = roc_curve(data['y_true'], data['y_proba'])
        ax.plot(fpr_v, tpr_v, color=COLORS[i%6], lw=2,
                label=f'{name} (AUC={auc(fpr_v,tpr_v):.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curves', fontweight='bold')
    ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

# 10c. Bar chart
fig, ax = plt.subplots(figsize=(14, 6))
bar_cols = ['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','Detection Rate (%)']
results_df[bar_cols].plot(kind='bar', ax=ax, color=COLORS[:5], width=0.75)
ax.set_ylabel('Score (%)'); ax.set_title('Model Comparison', fontweight='bold')
ax.set_ylim([max(results_df[bar_cols].min().min()-2, 80), 101])
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
for c in ax.containers: ax.bar_label(c, fmt='%.2f', fontsize=6, padding=2)
plt.tight_layout(); plt.show()

# 10d. Confusion matrices with class name labels (FIX D)
n_models = len(all_predictions)
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, (name, data) in enumerate(all_predictions.items()):
    ax  = axes[idx]
    cm  = confusion_matrix(data['y_true'], data['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)
    m = all_results[name]
    ax.set_title(
        f'{name}\nAcc={m["Accuracy (%)"]:.2f}%  F1={m["F1-Score (%)"]:.2f}%  FPR={m["FPR (%)"]:.2f}%',
        fontweight='bold', fontsize=10)
for idx in range(n_models, len(axes)): axes[idx].set_visible(False)
plt.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# 10e. Classification reports
print('\n' + '='*60 + '\n  CLASSIFICATION REPORTS\n' + '='*60)
for name, data in all_predictions.items():
    print(f'\n--- {name} ---')
    print(classification_report(data['y_true'], data['y_pred'],
          target_names=class_names, digits=4))

In [ ]:
# CELL 11: Learning Curves Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
LINE_STYLES = ['-','--','-.', ':', '-','--']
MARKERS     = ['o','s','^', 'D','v', 'P']
for (metric_key, title, ylabel, ax) in [
    ('loss',         'Training Loss',       'Loss',     axes[0,0]),
    ('val_loss',     'Validation Loss',     'Loss',     axes[0,1]),
    ('accuracy',     'Training Accuracy',   'Accuracy', axes[1,0]),
    ('val_accuracy', 'Validation Accuracy', 'Accuracy', axes[1,1]),
]:
    for i, (name, hist) in enumerate(all_histories.items()):
        ep = range(1, len(hist[metric_key])+1)
        ax.plot(ep, hist[metric_key], color=COLORS[i%6], lw=2,
                linestyle=LINE_STYLES[i%6], marker=MARKERS[i%6],
                markersize=4, markevery=max(1, len(hist[metric_key])//10),
                label=name, alpha=0.9)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Learning Curves — All Models (v9 TPU)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# CELL 12: SHAP Feature Importance
#
# T10: SHAP must run on a CPU-backed copy of the model.
# TPU ops (XLA-compiled) are not compatible with SHAP's numerical
# perturbation approach. We rebuild the model on CPU and copy weights.
import shap
import warnings
warnings.filterwarnings('ignore')

print('Rebuilding LSTM-CNN on CPU for SHAP...')

# Rebuild model on CPU (no strategy scope)
# Use a dummy total_steps — we only need predict(), not training
# JAX backend: use jax.default_device to pin the CPU-side SHAP model.
# tf.device() is a TF context and has no effect on JAX/Keras ops.
_cpu = jax.devices('cpu')[0]
with jax.default_device(_cpu):
    cpu_model = build_lstm_cnn(INPUT_SHAPE, total_steps=1)
    cpu_model.set_weights(trained_models['LSTM-CNN (Hybrid)'].get_weights())

N_BACKGROUND = min(100, len(X_test))
N_EXPLAIN    = min(200, len(X_test))

background = X_test[:N_BACKGROUND]
explain_X  = X_test[:N_EXPLAIN]

def predict_fn_cpu(X_2d):
    X_3d = X_2d.reshape(X_2d.shape[0], X_2d.shape[1], 1)
    with jax.default_device(jax.devices('cpu')[0]):
        proba = cpu_model.predict(X_3d, verbose=0)
    return proba.flatten() if IS_BINARY else proba

print('Computing SHAP values — this may take a few minutes...')
explainer   = shap.KernelExplainer(predict_fn_cpu, background)
shap_values = explainer.shap_values(explain_X, nsamples=100, l1_reg='aic')

pca_feature_names = [f'PC_{i+1}' for i in range(X_test.shape[1])]

# FIX F: aggregate SHAP correctly for multiclass
if isinstance(shap_values, list):
    sv = np.mean([np.abs(sv_c) for sv_c in shap_values], axis=0).mean(axis=0)
else:
    sv = np.abs(shap_values).mean(axis=0)

sorted_idx   = np.argsort(sv)[::-1][:20]
top_features = [pca_feature_names[i] for i in sorted_idx]
top_shap     = sv[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_shap, color='#2196F3', alpha=0.85)
ax.set_yticks(range(len(top_features))); ax.set_yticklabels(top_features)
ax.invert_yaxis(); ax.set_xlabel('Mean |SHAP value|')
ax.set_title('LSTM-CNN Hybrid — Top 20 Feature Importance (SHAP)', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 10 features:')
for i in range(min(10, len(top_features))):
    print(f'  {i+1:2d}. {top_features[i]}: {top_shap[i]:.4f}')

In [ ]:
# CELL 13: Save All Results
# T11: model.save() works locally on TPU v5e-1 (single-host).
# For multi-host TPU pods, replace local path with gs://your-bucket/path
import json
from datetime import datetime
from sklearn.metrics import classification_report   # FIX G

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR   = f'results/run_{timestamp}'
for d in [f'{RUN_DIR}/figures', f'{RUN_DIR}/tables', f'{RUN_DIR}/models', f'{RUN_DIR}/training_history']:
    os.makedirs(d, exist_ok=True)
print(f'Saving to: {RUN_DIR}/')

results_df.to_csv(f'{RUN_DIR}/tables/model_comparison.csv')
results_df.to_latex(f'{RUN_DIR}/tables/model_comparison.tex', float_format='%.2f')

run_meta = dict(
    timestamp=timestamp, sample_frac=SAMPLE_FRAC,
    scaler='MinMaxScaler[0,1]', scheduler='CosineDecay',
    epochs_max=EPOCHS, global_batch_size=GLOBAL_BATCH_SIZE,
    batch_per_core=BATCH_SIZE_PER_CORE, n_cores=N_CORES,
    precision=PRECISION_POLICY,
    task='binary' if IS_BINARY else 'multiclass',
    tpu_enabled=ON_TPU,
    version='v9-tpu',
)
full_results = {
    'run_config': run_meta,
    'metrics': {
        n: {k: float(v) if isinstance(v, (int, float, np.integer, np.floating)) else v
            for k, v in m.items()}
        for n, m in all_results.items()
    }
}
with open(f'{RUN_DIR}/tables/all_results.json', 'w') as f:
    json.dump(full_results, f, indent=2)

for name, hist in all_histories.items():
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    with open(f'{RUN_DIR}/training_history/{safe}_history.json', 'w') as f:
        json.dump({k: [float(v) for v in vals] for k, vals in hist.items()}, f, indent=2)

class_names = [str(c) for c in le.classes_]
with open(f'{RUN_DIR}/tables/classification_reports.txt', 'w') as f:
    for name, data in all_predictions.items():
        f.write(f'\n{"="*60}\n  {name}\n{"="*60}\n')
        f.write(classification_report(data['y_true'], data['y_pred'],
                target_names=class_names, digits=4))

feat_df = pd.DataFrame({'Feature': top_features, 'SHAP_importance': top_shap})
feat_df.to_csv(f'{RUN_DIR}/tables/shap_importance.csv', index=False)

for name, model in trained_models.items():
    safe = name.lower().replace(' ','_').replace('(','').replace(')','').replace('-','_')
    # T11: local save works for v5e-1 (single host)
    model.save(f'{RUN_DIR}/models/{safe}.keras')

print('All results saved.')
for root, dirs, files in os.walk(RUN_DIR):
    lvl = root.replace(RUN_DIR,'').count(os.sep)
    print('  '*lvl + os.path.basename(root) + '/')
    for fi in sorted(files):
        kb = os.path.getsize(os.path.join(root, fi)) / 1024
        print('  '*(lvl+1) + fi + f' ({kb:.1f} KB)')

## v9 TPU — Change Summary

| Change | v8 (GPU) | v9 TPU |
|--------|----------|--------|
| Hardware setup | `mixed_float16` + GPU memory growth | `TPUStrategy` + `bfloat16` |
| v5e chip architecture | N/A | 1 chip = 1 TensorCore; v5e-1 → N_CORES=1 (not 4) |
| Precision | `float16` | `bfloat16` (native on TPU) |
| Model building | Module scope | Inside `strategy.scope()` |
| Batch size | 128 global | 128 × N_cores global |
| `class_weight` | `model.fit(class_weight=...)` | `sample_weight` via dataset tuple |
| `drop_remainder` | absent | `True` on all datasets |
| Cosine decay closure | Global `TOTAL_STEPS` | Explicit `total_steps` argument |
| Val divergence | 0.003% val_acc | `val_sample_weight` fix |
| SHAP | On GPU model | CPU-rebuilt model copy |
| Model save | TPU-stream (may fail) | Local path (works on v5e-1) |

## Target Results (paper Table 6)

| Model | Accuracy | Precision | Recall | F1 | FPR |
|-------|----------|-----------|--------|----|-----|
| **LSTM-CNN** | **99.87%** | **99.89%** | **99.85%** | **99.87%** | **0.13%** |
| BiLSTM | 98.92% | 98.97% | 98.87% | 98.92% | 1.03% |
| LSTM | 98.34% | 98.42% | 98.26% | 98.34% | 1.58% |
| GRU | 97.89% | 97.95% | 97.83% | 97.89% | 2.05% |
| CNN | 97.45% | 97.62% | 97.28% | 97.45% | 2.38% |
| RNN | 95.78% | 95.91% | 95.65% | 95.78% | 4.09% |